# Clase 5 — Taller de cierre: Prompt Engineering integrado

A lo largo de las últimas cuatro clases trabajaste con las principales técnicas de ingeniería de prompts:
- **Anatomía del prompt** — rol, contexto, instrucción y formato de salida.
- **Errores y patrones** — reconocer la vaguedad, el sesgo de asunción y los patrones como *Persona* o *Plantilla*.
- **Zero-shot y few-shot** — cuándo alcanza una instrucción directa y cuándo necesitás ejemplos para guiar al modelo.
- **Chain of Thought (CoT)** — forzar un razonamiento explícito paso a paso.

En este taller vas a usar **todas esas herramientas sobre un mismo caso real**: la startup ficticia **NEXO Tech** y su aplicación de gestión de tareas para equipos remotos, *NexoDesk*.

Cada ejercicio mira el mismo producto desde un ángulo distinto. No hay respuestas únicas: el objetivo es que iterés tu prompt, observés el resultado y reflexionés sobre por qué funciona o no.

---

| Ejercicio | Técnica central | Foco |
|:---|:---|:---|
| **1 — La voz del asistente** | Anatomía del prompt + Rol | Definir el comportamiento del bot de soporte de NexoDesk. |
| **2 — Entender a los usuarios** | Few-Shot | Clasificar los mensajes de usuarios según su tipo. |
| **3 — Asesorar al equipo** | Chain of Thought | Razonar una decisión estratégica difícil paso a paso. |

---
## Configuración del entorno

Antes de empezar, ejecutá esta celda para cargar la API key y el cliente del modelo.

In [ ]:
BACKEND = "local"

In [ ]:
import os, getpass

GEMINI_MODEL = "gemini-2.5-flash-lite"

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
if not GEMINI_API_KEY and BACKEND == "gemini":
    GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini: ")

print(f"Backend: {BACKEND}")

In [ ]:
if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente = genai.Client(api_key=GEMINI_API_KEY)
elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    ruta_modelo = hf_hub_download(
        repo_id="unsloth/gemma-3-1b-it-GGUF",
        filename="gemma-3-1b-it-Q4_K_M.gguf"
    )
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)


def llamar_llm(prompt, system_prompt="Sos un asistente útil.", temperature=0.7, max_tokens=400):
    if BACKEND == "gemini":
        r = _cliente.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()
    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Todo listo.'", max_tokens=10))

---
## Ejercicio 1 — La voz del asistente

### El contexto
NexoDesk está por lanzar un chatbot de soporte dentro de su aplicación. Este bot debe:
- Resolver dudas sobre el producto (cómo crear tareas, agregar miembros a un proyecto, exportar reportes, etc.).
- Derivar a un humano cuando el problema requiere acción sobre la cuenta (pagos, cancelaciones, bugs críticos).
- Mantener un tono profesional pero cercano, acorde al espíritu de una startup.

### Tu tarea
Hacer que el bot derive a un humano cuando el mensaje del usuario lo requiera manipulando el **system prompt**.
- Redactá el **system prompt** del asistente de NexoDesk. 
- Definí las reglas para cuándo el bot debe derivar a un humano.
- Luego probá tu prompt con los tres mensajes de usuario que están debajo y observá si el bot responde de acuerdo a lo que definiste.

**Reflexioná:** ¿Qué pasa si omitís el rol? ¿Si usás un tono muy formal? ¿El bot sabe cuándo derivar?

> **Buenas prácticas para escribir un system prompt**
>
> | Práctica | Por qué importa |
> |:---|:---|
> | **Definí un rol explícito** | El modelo ajusta su vocabulario, tono y nivel de detalle según el personaje que le asignás. Sin rol, responde de forma genérica. |
> | **Sé específico con los límites** | Decile qué *no* debe hacer, no solo qué sí. "No respondas preguntas sobre la competencia" es más efectivo que "respondé solo sobre NexoDesk". |
> | **Usá ejemplos dentro del prompt** | Si querés un tono particular, mostralo con una frase de muestra. El modelo imita mejor que interpreta. |
> | **Separé el rol del contexto** | El rol define *quién es* el bot; el contexto define *dónde está*. Mezclarlos genera instrucciones difusas. |
> | **Versioná y comparás** | Cambiá una sola cosa por vez. Si modificás el tono y la longitud al mismo tiempo, no sabrás qué generó el cambio. |

In [ ]:
# ── Tu system prompt ──────────────────────────────────────────────────────────
mi_system_prompt = """
# TODO: escribí aquí el system prompt del asistente de NexoDesk.
# Acordate de definir: rol, nombre (si querés), tono, qué puede y qué no puede hacer.
"""

# ── Mensajes de prueba (no modificar) ─────────────────────────────────────────
mensajes_de_prueba = [
    "Hola, ¿cómo hago para agregar a alguien de mi equipo a un proyecto que ya creé?",
    "Quiero cancelar mi suscripción y que me devuelvan lo que pagué este mes.",
    "¿Cuál es el mejor equipo de fútbol del mundo?",
]

# ── Ejecutar y comparar ───────────────────────────────────────────────────────
for i, mensaje in enumerate(mensajes_de_prueba, 1):
    print(f"[Mensaje {i}] {mensaje}")
    print("-" * 50)
    respuesta = llamar_llm(mensaje, system_prompt=mi_system_prompt)
    print(respuesta)
    print()

### Preguntas para la reflexión
1. ¿En cuál de los tres mensajes tu bot funcionó mejor? ¿Por qué?
2. ¿Qué le faltaba a tu primer intento? ¿Qué cambiaste para mejorarlo?
3. ¿Cómo indicaste (o deberías indicar) el límite entre lo que el bot resuelve y lo que deriva a un humano?

---
## Ejercicio 2 — Entender a los usuarios

### El contexto
El equipo de producto de NexoDesk recibe decenas de mensajes de usuarios por día. Muchos llegan mezclados al mismo canal: algunos reportan un bug, otros piden una nueva funcionalidad, otros se quejan del precio y otros simplemente felicitan al equipo.

Para priorizar el trabajo, necesitan clasificar cada mensaje en una de estas cuatro categorías:
- **Bug** — algo no funciona como debería.
- **Sugerencia** — el usuario propone una mejora o un feature nuevo.
- **Elogio** — feedback positivo.
- **Queja comercial** — problemas relacionados con precio, suscripción o atención.

### Tu tarea
Diseñá un prompt **few-shot** que clasifique correctamente los cuatro mensajes de prueba. Vas a necesitar incluir ejemplos propios en tu prompt para enseñarle al modelo el patrón que buscás.

**Clave:** el modelo no sabe qué categorías existen ni cómo reconocerlas. Los ejemplos que pongas son lo que definen su criterio. Elegí bien.

In [ ]:
# ── Mensajes de prueba (no modificar) ─────────────────────────────────────────
mensajes_feedback = [
    "Cada vez que intento exportar el reporte en PDF, la app se cierra. Probé en Chrome y en Firefox.",
    "Estaría buenísimo que se pudiese arrastrar y soltar las tareas entre columnas, como en Trello.",
    "¡Felicitaciones al equipo! Desde que empezamos a usar NexoDesk las reuniones son mucho más cortas.",
    "El plan Pro cuesta el doble que la competencia y tiene menos funciones. No lo renuevo.",
]

# ── Tu prompt few-shot ─────────────────────────────────────────────────────────
# TODO: completá la cadena de texto con tus ejemplos de clasificación.
# Cada ejemplo debe mostrar un mensaje y su categoría, para que el modelo imite el patrón.
mi_prompt_few_shot = """

# TODO: ponés aquí tus ejemplos (few-shot).

Mensaje: {mensaje}

Categoría:
"""

# ── Ejecutar ──────────────────────────────────────────────────────────────────
for i, msg in enumerate(mensajes_feedback, 1):
    prompt = mi_prompt_few_shot.format(mensaje=msg)
    resultado = llamar_llm(prompt, temperature=0.1, max_tokens=80)
    print(f"[Mensaje {i}] {msg[:70]}...")
    print(f"  → {resultado}")
    print()

### Preguntas para la reflexión
1. ¿Cuántos ejemplos fueron necesarios para que el modelo clasificara bien los cuatro casos?
2. ¿Qué pasó con los mensajes más ambiguos (por ejemplo, uno que podría ser tanto Sugerencia como Queja)?
3. ¿Cómo cambiaría tu prompt si hubiera una quinta categoría, "Pregunta técnica"?

---
## Ejercicio 3 — Asesorar al equipo

### El contexto
El equipo de NexoDesk tiene que tomar una decisión difícil esta semana. Los datos de la app muestran lo siguiente:

- El **67 % de los usuarios que cancelaron** en el último trimestre mencionaron la falta de un **modo sin conexión a internet** como su razón principal.
- Implementar el modo offline correctamente requiere al menos **3 meses** de trabajo de ingeniería, según la estimación del CTO.
- El inversor principal tiene una reunión de resultados en **5 semanas** y espera ver una funcionalidad nueva relevante para los usuarios.
- El equipo de diseño tiene lista en dos semanas una propuesta de **rediseño del tablero de tareas** (menos impacto pero más rápida de desarrollar).

Hay dos caminos posibles:
- **Opción A:** empezar con el modo offline aunque no esté listo para la reunión del inversor.
- **Opción B:** lanzar el rediseño del tablero para la reunión y posponer el modo offline.

### Tu tarea
Diseñá un prompt que use **Chain of Thought (CoT)** para que el modelo razone esta decisión paso a paso y llegue a una recomendación fundamentada.

Después, probá el **mismo prompt pero sin instrucción de CoT** y comparás los resultados.

In [ ]:
contexto_nexo = """
Contexto:
- El 67% de los usuarios que se dieron de baja mencionaron la falta de modo offline.
- Implementar modo offline: 3 meses de desarrollo (estimación del CTO).
- Reunión con el inversor principal: en 5 semanas, espera ver una funcionalidad nueva relevante.
- Rediseño del tablero de tareas: listo en 2 semanas, menor impacto percibido por usuarios.

Opciones:
- Opción A: arrancar con el modo offline aunque no esté listo para la reunión.
- Opción B: lanzar el rediseño del tablero para la reunión y posponer el modo offline.
"""

# ── Prompt SIN Chain of Thought (línea base) ──────────────────────────────────
prompt_directo = f"""{contexto_nexo}
¿Qué opción debería elegir el equipo de NexoDesk? Dame una recomendación.
"""

print("=" * 60)
print("RESPUESTA SIN CoT (respuesta directa)")
print("=" * 60)
print(llamar_llm(prompt_directo, temperature=0.3))
print()

# ── Tu prompt CON Chain of Thought ───────────────────────────────────────────
# TODO: redactá aquí la instrucción de CoT que le pida al modelo razonar
# explícitamente antes de responder (paso a paso, pros y contras, contexto, etc.)
mi_prompt_cot = f"""{contexto_nexo}
# TODO: tu instrucción de CoT va aquí.
"""

print("=" * 60)
print("RESPUESTA CON CoT")
print("=" * 60)
print(llamar_llm(mi_prompt_cot, temperature=0.3, max_tokens=600))

### Preguntas para la reflexión
1. ¿En qué se diferenciaron las dos respuestas (con y sin CoT)?
2. ¿La respuesta con CoT consideró factores que la directa ignoró?
3. ¿Cambiaría tu prompt si el destinatario de la recomendación fuese el inversor y no el equipo interno?